# **LAB — Bagging e Boosting com o dataset barley**

Nomes: Caroline Guimarães Campos (RA00335765)
       Isabela Groke Gomes (RA00332551)
Implementar e comparar um modelo de Bagging e um modelo de Boosting, usando Scikit-Learn e o conjunto seattle_weather do pacote vega_datasets.

Como a variável escolhida é numérica, o código utiliza as versões de regressão dos estimadores: BaggingRegressor e GradientBoostingRegressor.

### **1. Importar as bibliotecas**

In [1]:
import pandas as pd

from vega_datasets import data
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### **2. Carregar o dataset seattle_weather**

In [5]:
df = data.seattle_weather()

print("Dimensões do dataset:", df.shape)
display(df.head())

print("\nColunas:")
print(df.columns.tolist())

print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isna().sum())

Dimensões do dataset: (1461, 6)


,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain



Colunas:
['date', 'precipitation', 'temp_max', 'temp_min', 'wind', 'weather']

Tipos de dados:
date             datetime64[ns]
precipitation           float64
temp_max                float64
temp_min                float64
wind                    float64
weather                  object
dtype: object

Valores ausentes por coluna:
date             0
precipitation    0
temp_max         0
temp_min         0
wind             0
weather          0
dtype: int64


### **3. Preparar os dados**
A variável que queremos prever é a temp_max, que indica a temperatura máxima de cada dia. A partir da coluna date, extraímos o ano, o mês e o dia. As outras colunas serão utilizadas como informações para ajudar o modelo a fazer as previsões.

Como a coluna weather contém categorias de clima, usamos o OneHotEncoder para transformá-las em valores que o modelo consiga interpretar. Por fim, dividimos os dados em duas partes: 80% para treinar o modelo e 20% para testar seu desempenho.

In [6]:
# Converter a coluna de data e criar características numéricas de calendário
df = df.copy()
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df = df.drop(columns=["date"])

# Remover linhas sem alvo e sem dados de entrada
df = df.dropna(subset=["temp_max"]).copy()

X = df.drop(columns=["temp_max"])
y = df["temp_max"]

colunas_categoricas = X.select_dtypes(include=["object", "category"]).columns.tolist()
colunas_numericas = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

preprocessador = ColumnTransformer(transformers=[("categoricas", OneHotEncoder(handle_unknown="ignore"), colunas_categoricas), ("numericas", "passthrough", colunas_numericas)])

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.20, random_state=42)

print("Quantidade de exemplos para treino:", len(X_treino))
print("Quantidade de exemplos para teste:", len(X_teste))
print("Variáveis categóricas:", colunas_categoricas)
print("Variáveis numéricas:", colunas_numericas)

Quantidade de exemplos para treino: 1168
Quantidade de exemplos para teste: 293
Variáveis categóricas: ['weather']
Variáveis numéricas: ['precipitation', 'temp_min', 'wind', 'year', 'month', 'day']


### **4. Modelo de Bagging**
O Bagging funciona treinando vários modelos com diferentes amostras dos dados de treinamento e combinando suas previsões para obter um resultado mais confiável. Neste caso, utilizamos árvores de decisão como modelos-base e o BaggingRegressor para prever a temperatura máxima diária.

In [8]:
modelo_bagging = Pipeline(steps=[("preprocessamento", preprocessador),
    ("modelo", BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42), n_estimators=100, random_state=42))])

modelo_bagging.fit(X_treino, y_treino)
previsoes_bagging = modelo_bagging.predict(X_teste)

### **5. Modelo de Boosting**
O Boosting treina estimadores sequencialmente, buscando melhorar o resultado a cada etapa. Nesta implementação, usamos GradientBoostingRegressor.

In [9]:
modelo_boosting = Pipeline(steps=[("preprocessamento", preprocessador),
    ("modelo", GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))])

modelo_boosting.fit(X_treino, y_treino)
previsoes_boosting = modelo_boosting.predict(X_teste)

### **6. Avaliar e comparar os modelos**
Usamos três métricas: MAE (erro absoluto médio), RMSE (raiz do erro quadrático médio) e R².

Para MAE e RMSE, valores menores indicam erros menores.

Para R², valores mais próximos de 1 indicam melhor ajuste.

In [10]:
def avaliar_modelo(nome, valores_reais, previsoes):
    return {"Modelo": nome, "MAE": mean_absolute_error(valores_reais, previsoes), "RMSE": mean_squared_error(valores_reais, previsoes) ** 0.5, "R²": r2_score(valores_reais, previsoes)}

resultados = pd.DataFrame([avaliar_modelo("Bagging", y_teste, previsoes_bagging), avaliar_modelo("Boosting", y_teste, previsoes_boosting),])

display(resultados.round(3))

,Modelo,MAE,RMSE,R²
0,Bagging,2.030,2.615,0.88
1,Boosting,2.026,2.608,0.88


### **7. Visualizar valores reais e previstos**
A tabela abaixo permite comparar, para cada exemplo do conjunto de teste, o temperatura máxima real com as previsões feitas pelos dois modelos.

In [11]:
comparacao = pd.DataFrame({"Valor real (temp_max)": y_teste.to_numpy(), "Previsão Bagging": previsoes_bagging, "Previsão Boosting": previsoes_boosting})

display(comparacao.head(10).round(3))

,Valor real (temp_max),Previsão Bagging,Previsão Boosting
0,23.9,22.147,22.356
1,7.8,9.985,9.196
2,9.4,11.622,11.756
3,26.7,23.734,24.266
4,13.3,11.429,11.245
5,21.7,22.974,21.636
6,28.3,27.662,28.334
7,15.0,12.991,13.753
8,13.9,17.728,15.936
9,25.0,20.358,21.659
